# DDRI Bike-Change rep15 데이터셋 빌더

## 용어 설명

- `bike_change`(시간별 대여량 변화량): 현재 시간 `rental_count`에서 직전 시간 `rental_count`를 뺀 값
- `feature`(입력 변수): 모델이 예측에 사용하는 입력값
- `lag`(과거 시점 값): 직전 몇 시간, 며칠 전 값을 가져온 피처
- `rolling`(이동 통계): 최근 일정 구간의 평균, 표준편차 같은 통계 피처
- `cluster`(군집 라벨): 대여소를 이용 패턴별로 묶은 보조 피처

이번 노트북의 가정:
- 현재 `rep15` 입력에는 실제 재고 수량이나 반납 수가 없으므로, 1차 실험에서는 `bike_change`를 `rental_count`의 시간별 변화량으로 정의한다.
- full161 필터링 실험의 `top20_optional` 38개 중 `morning_net_inflow`, `evening_net_inflow` 2개는 rep15 입력에 없어서 제외한다.
- 따라서 rep15 1차 실험은 `36개 사용 가능 피처`로 시작한다.


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
OUTPUT_DATA_DIR = WORK_DIR / 'output/data'

TRAIN_PATH = ROOT / '3조 공유폴더/대표대여소_예측데이터_15개/second_round_data/ddri_prediction_long_train_2023_2024_second_round_feature_collection.csv'
TEST_PATH = ROOT / '3조 공유폴더/대표대여소_예측데이터_15개/second_round_data/ddri_prediction_long_test_2025_second_round_feature_collection.csv'

TRAIN_OUT = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_train_2023_2024.csv'
TEST_OUT = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_test_2025.csv'
FEATURE_SUMMARY_OUT = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_feature_summary.csv'


In [2]:
BASE_FEATURES = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed'
]

DERIVED_TIME_SERIES = [
    'lag_1h', 'lag_24h', 'lag_168h',
    'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_168h', 'rolling_std_168h'
]

AVAILABLE_OPTIONAL = [
    'rolling_mean_6h', 'is_weekend', 'is_night_hour', 'is_commute_hour',
    'hour_sin', 'is_rainy', 'hour_cos', 'commute_morning_flag',
    'commute_evening_flag', 'subway_distance_m', 'distance_naturepark_m',
    'restaurant_count_300m', 'convenience_store_count_300m', 'bus_stop_count_300m',
    'cafe_count_300m', 'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]

TARGET_FEATURES = BASE_FEATURES + DERIVED_TIME_SERIES + AVAILABLE_OPTIONAL

FEATURE_NAME_KO = {
    'station_id': '대여소 ID',
    'cluster': '군집 라벨',
    'mapped_dong_code': '행정동 코드',
    'hour': '시각',
    'weekday': '요일',
    'month': '월',
    'holiday': '공휴일 여부',
    'temperature': '기온',
    'humidity': '습도',
    'precipitation': '강수량',
    'wind_speed': '풍속',
    'lag_1h': '1시간 전 대여량',
    'lag_24h': '24시간 전 대여량',
    'lag_168h': '168시간 전 대여량',
    'rolling_mean_24h': '최근 24시간 대여량 평균',
    'rolling_std_24h': '최근 24시간 대여량 표준편차',
    'rolling_mean_168h': '최근 168시간 대여량 평균',
    'rolling_std_168h': '최근 168시간 대여량 표준편차',
    'rolling_mean_6h': '최근 6시간 대여량 평균',
    'is_weekend': '주말 여부',
    'is_night_hour': '야간 시간대 여부',
    'is_commute_hour': '출퇴근 시간대 여부',
    'hour_sin': '시각 사인값',
    'is_rainy': '비 여부',
    'hour_cos': '시각 코사인값',
    'commute_morning_flag': '아침 출근 시간대 여부',
    'commute_evening_flag': '저녁 퇴근 시간대 여부',
    'subway_distance_m': '가까운 지하철역까지 거리',
    'distance_naturepark_m': '자연공원까지 거리',
    'restaurant_count_300m': '300m 안 음식점 수',
    'convenience_store_count_300m': '300m 안 편의점 수',
    'bus_stop_count_300m': '300m 안 버스정류장 수',
    'cafe_count_300m': '300m 안 카페 수',
    'elevation_diff_nearest_subway_m': '가까운 지하철역과의 고도 차',
    'nearest_park_area_sqm': '가까운 공원 면적',
    'bike_change': '시간별 대여량 변화량'
}

len(TARGET_FEATURES)


35

In [3]:
def add_rental_timeseries_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result = result.sort_values(['station_id', 'date', 'hour']).reset_index(drop=True)
    grouped = result.groupby('station_id')['rental_count']
    shifted = grouped.shift(1)

    result['bike_change'] = grouped.diff().astype('float32')
    result['commute_morning_flag'] = result['hour'].isin([7, 8, 9]).astype('int8')
    result['commute_evening_flag'] = result['hour'].isin([17, 18, 19]).astype('int8')
    result['lag_1h'] = grouped.shift(1).astype('float32')
    result['lag_24h'] = grouped.shift(24).astype('float32')
    result['lag_168h'] = grouped.shift(168).astype('float32')
    result['rolling_mean_24h'] = shifted.rolling(24).mean().reset_index(level=0, drop=True).astype('float32')
    result['rolling_std_24h'] = shifted.rolling(24).std().reset_index(level=0, drop=True).astype('float32')
    result['rolling_mean_168h'] = shifted.rolling(168).mean().reset_index(level=0, drop=True).astype('float32')
    result['rolling_std_168h'] = shifted.rolling(168).std().reset_index(level=0, drop=True).astype('float32')
    return result

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_feat = add_rental_timeseries_features(train_raw)
test_feat = add_rental_timeseries_features(test_raw)

train_feat.shape, test_feat.shape


((263160, 61), (131400, 61))

In [4]:
required_cols = ['station_name', 'station_group', 'date', 'rental_count', 'bike_change'] + TARGET_FEATURES

train_dataset = train_feat[required_cols].dropna().reset_index(drop=True)
test_dataset = test_feat[required_cols].dropna().reset_index(drop=True)

train_dataset.to_csv(TRAIN_OUT, index=False, encoding='utf-8-sig')
test_dataset.to_csv(TEST_OUT, index=False, encoding='utf-8-sig')

feature_summary = pd.DataFrame({
    'feature': TARGET_FEATURES,
    'feature_ko': [FEATURE_NAME_KO[f] for f in TARGET_FEATURES],
    'exists_in_rep15_second_round_input': [f in train_raw.columns for f in TARGET_FEATURES],
    'derived_in_notebook': [f in DERIVED_TIME_SERIES for f in TARGET_FEATURES],
})
feature_summary.to_csv(FEATURE_SUMMARY_OUT, index=False, encoding='utf-8-sig')

print('saved:', TRAIN_OUT)
print('saved:', TEST_OUT)
print('saved:', FEATURE_SUMMARY_OUT)
pd.DataFrame({
    'dataset': ['train', 'test'],
    'rows': [len(train_dataset), len(test_dataset)],
    'columns': [len(train_dataset.columns), len(test_dataset.columns)],
})


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_prediction_bike_change_train_2023_2024.csv
saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_prediction_bike_change_test_2025.csv
saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_prediction_bike_change_feature_summary.csv


,dataset,rows,columns
0,train,260640,40
1,test,128880,40


In [5]:
feature_summary


,feature,feature_ko,exists_in_rep15_second_round_input,derived_in_notebook
0,station_id,대여소 ID,True,False
1,cluster,군집 라벨,True,False
2,mapped_dong_code,행정동 코드,True,False
3,hour,시각,True,False
4,weekday,요일,True,False
5,month,월,True,False
6,holiday,공휴일 여부,True,False
7,temperature,기온,True,False
8,humidity,습도,True,False
9,precipitation,강수량,True,False
